<a href="https://colab.research.google.com/github/Alexandr20i/sirius-python-course/blob/main/lab_02_python_%D0%91%D0%B5%D0%BB%D0%BE%D0%B2%D0%90%D0%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №2
## Коллекции и управление программой: платёжный шлюз

### Итог работы

Вы обработаете пакет платёжных событий: найдёте дубликаты и
некорректные записи, нормализуете данные, примените правила
платёжного шлюза, измените балансы, сформируете очередь ручной
проверки и построите отчёт смены. В финале нужно самостоятельно
реализовать второй сценарий с дополнительным velocity-лимитом.

### Новые инструменты

`list`, `tuple`, `dict`, `set`, индексы и срезы, `for`, `while`,
`if`/`elif`/`else`, `match`, `break`, `continue`, comprehensions,
`enumerate`, методы коллекций и сортировка.

Функции, классы, файлы, NumPy и Pandas пока не используются. Это
намеренное ограничение: цель работы — увидеть состояние алгоритма
и движение данных внутри циклов.

### Правило выполнения

Запускайте ячейки сверху вниз и не изменяйте исходные наборы
`raw_events` и `capstone_events`. Перед сдачей выполните **Restart
Kernel and Run All Cells**. Все `assert` должны пройти.


## 1. Выбор коллекции

| Структура | Что хранит | Когда выбирать |
|---|---|---|
| `list` | упорядоченные элементы, повторы разрешены | очередь и журнал |
| `tuple` | неизменяемую последовательность | границы и составной ключ |
| `dict` | пары ключ → значение | баланс по номеру счёта |
| `set` | уникальные элементы | дубликаты и стоп-листы |

В этой работе одна и та же операция представлена словарём, пакет
операций — списком, стоп-лист — множеством, а границы пакета —
кортежем.


In [ ]:
initial_balances = {
    "A100": 25_000.0,
    "A200": 15_000.0,
    "A300": 7_000.0,
    "A400": 5_000.0,
}
daily_limits = {
    "A100": 12_000.0,
    "A200": 8_000.0,
    "A300": 5_000.0,
    "A400": 3_000.0,
}
blocked_accounts = {"A400"}
blocked_merchants = {"SCAM-SHOP"}

raw_events = [
    {"id": "E001", "kind": "deposit", "account": "A100", "amount": 5000.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A100", "amount": 3200.0,
     "merchant": " books ", "city": "moscow", "counterparty": None},
    {"id": "E003", "kind": "purchase", "account": "A200", "amount": 6500.0,
     "merchant": "tech", "city": " kazan ", "counterparty": None},
    {"id": "E004", "kind": "purchase", "account": "A200", "amount": 2500.0,
     "merchant": "CAFE", "city": "Kazan", "counterparty": None},
    {"id": "E005", "kind": "withdrawal", "account": "A300", "amount": 1500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E006", "kind": "purchase", "account": "A400", "amount": 1000.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E007", "kind": "refund", "account": "A100", "amount": 700.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E008", "kind": "purchase", "account": "A100", "amount": 450.0,
     "merchant": "SCAM-SHOP", "city": "Moscow", "counterparty": None},
    {"id": "E009", "kind": "transfer", "account": "A300", "amount": 4000.0,
     "merchant": None, "city": "Moscow", "counterparty": "A200"},
    {"id": "E010", "kind": "cashout", "account": "A300", "amount": 500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A300", "amount": 50.0,
     "merchant": "CAFE", "city": "Moscow", "counterparty": None},
    {"id": "E012", "kind": "purchase", "account": "A100", "amount": -200.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E013", "kind": "purchase", "account": "A999", "amount": 100.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
]

# В записях только скалярные значения, поэтому для контроля
# неизменности достаточно скопировать каждый внутренний словарь.
raw_snapshot = [event.copy() for event in raw_events]
print(f"Получено событий: {len(raw_events)}")


Получено событий: 13


### Задание 1. Инвентаризация пакета

Не изменяя `raw_events`, получите:

- `event_ids` — список ID в исходном порядке;
- `unique_event_ids` — множество уникальных ID;
- `duplicate_ids` — множество повторяющихся ID;
- `accounts_seen` — множество всех встреченных счетов;
- `batch_bounds` — кортеж из первого и последнего ID;
- `first_three_ids` — первые три ID с помощью среза.

Здесь намеренно используйте обычные циклы: важно увидеть, как
постепенно заполняется изменяемая коллекция.


In [ ]:
event_ids = []
unique_event_ids = set()
duplicate_ids = set()
accounts_seen = set()
batch_bounds = None
first_three_ids = []
# YOUR CODE HERE

for ids in raw_events:
  event_ids.append(ids["id"])
  if ids["id"] in unique_event_ids:
      duplicate_ids.add(ids["id"])
  else:
      unique_event_ids.add(ids["id"])
  accounts_seen.add(ids["account"])
batch_bounds = (event_ids[0], event_ids[-1])
first_three_ids = event_ids[:3]

# print(event_ids)
# print(unique_event_ids)
# print(duplicate_ids)
# print(accounts_seen)
# print(batch_bounds)
# print(first_three_ids)


## 2. Контроль качества данных

Внешний пакет нельзя сразу применять к балансам. Сначала нужно
отделить структурные ошибки от бизнес-решений. Оператор `continue`
завершает текущую итерацию и переходит к следующей записи — это
удобно для отбраковки.

Не удаляйте элементы из списка во время `for`: индексы сдвигаются,
и часть записей легко пропустить. Вместо этого формируйте новый
список `clean_events`.


### Задание 2. Валидация и нормализация

Обойдите `raw_events` по порядку. Для каждой записи примените первую
подходящую причину брака:

1. повторный ID → `duplicate_id`;
2. `amount` не является обычным `int`/`float` или не положителен →
   `invalid_amount`;
3. счёт отсутствует в `initial_balances` → `unknown_account`;
4. у перевода неизвестный получатель → `unknown_counterparty`.

Некорректные записи добавляйте в `invalid_events` в формате
`{"id": ..., "reason": ...}`. Корректную запись сначала копируйте,
затем приведите `kind`, `merchant` и `city` к верхнему регистру без
внешних пробелов. `None` у merchant должен остаться `None`.

Заполните `seen_ids`, `clean_events` и `invalid_events`. Исходный
пакет должен остаться неизменным.


In [ ]:
seen_ids = set()
clean_events = []
invalid_events = []
# YOUR CODE HERE

for ids in raw_events:

  if ids["id"] in seen_ids:
    invalid_events.append({"id": ids["id"], "reason": "duplicate_id"})
    continue
  else:
    seen_ids.add(ids["id"])

  if not isinstance(ids["amount"], (int,float)) or ids["amount"] <= 0:
    invalid_events.append({"id": ids["id"], "reason": "invalid_amount"})
    continue

  if not ids["account"] in initial_balances:
    invalid_events.append({"id": ids["id"], "reason": "unknown_account"})
    continue

  if ids["kind"] == "transfer" and ids["counterparty"] not in initial_balances:
    invalid_events.append({"id": ids["id"], "reason": "unknown_counterparty"})
    continue

  clean_event = ids.copy()
  clean_event["kind"] = clean_event["kind"].upper().strip()
  if clean_event["merchant"] is not None:
    clean_event["merchant"] = clean_event["merchant"].upper().strip()
  if clean_event["city"] is not None:
    clean_event["city"] = clean_event["city"].upper().strip()

  clean_events.append(clean_event)


  print(ids)

print(invalid_events)
print(clean_events)

{'id': 'E001', 'kind': 'deposit', 'account': 'A100', 'amount': 5000.0, 'merchant': None, 'city': 'Moscow', 'counterparty': None}
{'id': 'E002', 'kind': 'purchase', 'account': 'A100', 'amount': 3200.0, 'merchant': ' books ', 'city': 'moscow', 'counterparty': None}
{'id': 'E003', 'kind': 'purchase', 'account': 'A200', 'amount': 6500.0, 'merchant': 'tech', 'city': ' kazan ', 'counterparty': None}
{'id': 'E004', 'kind': 'purchase', 'account': 'A200', 'amount': 2500.0, 'merchant': 'CAFE', 'city': 'Kazan', 'counterparty': None}
{'id': 'E005', 'kind': 'withdrawal', 'account': 'A300', 'amount': 1500.0, 'merchant': None, 'city': 'Moscow', 'counterparty': None}
{'id': 'E006', 'kind': 'purchase', 'account': 'A400', 'amount': 1000.0, 'merchant': 'BOOKS', 'city': 'Moscow', 'counterparty': None}
{'id': 'E007', 'kind': 'refund', 'account': 'A100', 'amount': 700.0, 'merchant': 'BOOKS', 'city': 'Moscow', 'counterparty': None}
{'id': 'E008', 'kind': 'purchase', 'account': 'A100', 'amount': 450.0, 'merch

## 3. Comprehensions и производные коллекции

Comprehension подходит для короткого преобразования или отбора без
сложного состояния. Если внутри требуется несколько веток,
изменение баланса или объяснение причины решения, обычный цикл
читается лучше.

Общая форма: `[выражение for элемент in источник if условие]`.
Аналогично создаются множества и словари.


### Задание 3. Операционный обзор

По `clean_events` создайте:

- `active_accounts` — отсортированный список незаблокированных
  счетов с положительным балансом;
- `merchant_catalog` — множество непустых merchant;
- `high_value_ids` — ID событий на сумму не меньше 5000 в исходном
  порядке;
- `kind_counts` — словарь количества событий каждого вида.

Первые три результата получите comprehensions. Для `kind_counts`
сначала создайте словарь с нулями, затем накопите значения циклом.


In [ ]:
active_accounts = []
merchant_catalog = set()
high_value_ids = []
kind_counts = {}
# YOUR CODE HERE

active_accounts = sorted({
    ids["account"]
    for ids in clean_events
    if ids["account"] not in blocked_accounts
    and initial_balances.get(ids["account"], 0) > 0
})

merchant_catalog = {
    ids["merchant"]
    for ids in clean_events
    if ids["merchant"] is not None
}

high_value_ids = [
    ids["id"]
    for ids in clean_events
    if ids["amount"] >= 5000
]

for ids in clean_events:
  kind_counts[ids["kind"]] = kind_counts.get(ids["kind"], 0) + 1
print(kind_counts)

{'DEPOSIT': 1, 'PURCHASE': 5, 'WITHDRAWAL': 1, 'REFUND': 1, 'TRANSFER': 1, 'CASHOUT': 1}


## 4. Состояние и порядок правил

Теперь каждая операция зависит от результатов предыдущих. Это
**состояние-зависимый алгоритм**: перестановка событий может изменить
итоговые балансы и решения.

Порядок проверок является частью спецификации:

1. заблокированный счёт;
2. запрещённый merchant;
3. поддерживаемый вид операции;
4. дневной лимит для списаний;
5. достаточный баланс;
6. применение операции.

`match` определяет смысл вида операции, а `if` проверяет правила.
Сумма перевода списывается у отправителя и зачисляется получателю.
Операции со статусом `REVIEW` и `REJECTED` пока не меняют баланс.


### Задание 4. Процессор платёжных событий

Создайте независимые копии балансов и нулевой расход по каждому
счёту. Обработайте все `clean_events`.

Для каждого события добавьте в `decisions` словарь с ключами:
`id`, `account`, `kind`, `amount`, `status`, `reason`,
`balance_after`.

Возможные статусы: `APPROVED`, `REVIEW`, `REJECTED`. Причины:
`ok`, `blocked_account`, `blocked_merchant`, `unsupported_kind`,
`daily_limit`, `insufficient_funds`.

События `REVIEW` (если превышен дневной лимит) копируйте в `review_queue`. Дневной расход
увеличивают только одобренные `purchase`, `withdrawal`, `transfer`.
`deposit` и `refund` увеличивают баланс и не расходуют лимит.


In [ ]:
final_balances = initial_balances.copy()
daily_spent = {account: 0.0 for account in initial_balances}
decisions = []
review_queue = []
# YOUR CODE HERE
# print(final_balances)


for ids in clean_events:
  if ids["account"] in blocked_accounts:
    decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "REJECTED",
        "reason": "blocked_account",
        "balance_after": final_balances[ids["account"]]
    })
    continue

  if ids["merchant"] is not None and ids["merchant"] in blocked_merchants:
    decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "REJECTED",
        "reason": "blocked_merchant",
        "balance_after": final_balances[ids["account"]]
    })
    continue

  if ids["kind"] not in ["DEPOSIT", "PURCHASE", "REFUND", "TRANSFER", "CASHOUT"]:
    decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "REJECTED",
        "reason": "unsupported_kind",
        "balance_after": final_balances[ids["account"]]
    })
    continue

  if ids["kind"] in ["WITHDRAWAL", "PURCHASE", "TRANSFER"]:
    if daily_spent[ids["account"]] + ids["amount"] > daily_limits[ids["account"]]:
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "REVIEW",
        "reason": "daily_limit",
        "balance_after": final_balances[ids["account"]]
      })
      review_queue.append(ids.copy())
      continue

  if final_balances[ids["account"]] < ids["amount"]:
    decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "REJECTED",
        "reason": "insufficient_funds",
        "balance_after": final_balances[ids["account"]]
    })
    continue

  match ids["kind"]:
    case "DEPOSIT":
      final_balances[ids["account"]] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

    case "REFUND":
      final_balances[ids["account"]] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

    case "PURCHASE":
      final_balances[ids["account"]] -= ids["amount"]
      daily_spent[ids['account']] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

    case "WITHDRAWAL":
      final_balances[ids["account"]] -= ids["amount"]
      daily_spent[ids['account']] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

    case "TRANSFER":
      final_balances[ids["account"]] -= ids["amount"]
      final_balances[ids["counterparty"]] +=ids["amount"]
      daily_spent[ids['account']] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

    case "CASHOUT":
      final_balances[ids["account"]] -= ids["amount"]
      # daily_spent[ids['account']] += ids["amount"]
      decisions.append({
        "id": ids["id"],
        "account": ids["account"],
        "kind": ids["kind"],
        "amount": ids["amount"],
        "status": "APPROVED",
        "reason": "ok",
        "balance_after": final_balances[ids["account"]]
      })

# print(clean_events)
# print(daily_spent)
print(decisions)
# print(review_queue)
# print(final_balances)

[{'id': 'E001', 'account': 'A100', 'kind': 'DEPOSIT', 'amount': 5000.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 30000.0}, {'id': 'E002', 'account': 'A100', 'kind': 'PURCHASE', 'amount': 3200.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 26800.0}, {'id': 'E003', 'account': 'A200', 'kind': 'PURCHASE', 'amount': 6500.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 8500.0}, {'id': 'E004', 'account': 'A200', 'kind': 'PURCHASE', 'amount': 2500.0, 'status': 'REVIEW', 'reason': 'daily_limit', 'balance_after': 8500.0}, {'id': 'E005', 'account': 'A300', 'kind': 'WITHDRAWAL', 'amount': 1500.0, 'status': 'REJECTED', 'reason': 'unsupported_kind', 'balance_after': 7000.0}, {'id': 'E006', 'account': 'A400', 'kind': 'PURCHASE', 'amount': 1000.0, 'status': 'REJECTED', 'reason': 'blocked_account', 'balance_after': 5000.0}, {'id': 'E007', 'account': 'A100', 'kind': 'REFUND', 'amount': 700.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 27500.0}, {'id': 'E008

## 5. Вложенная агрегация

Журнал решений удобен для аудита, но руководителю нужны итоги.
Словарь может содержать другие словари: например,
`stats_by_status[status]["amount"]`. Метод `setdefault` создаёт
начальное значение только при отсутствии ключа.


### Задание 5. Статистика решений


За один проход по `decisions` постройте:

- `stats_by_status`: для каждого статуса количество `count` и сумма
  `amount`;
- `reason_counts`: количество решений по каждой причине;
- `review_accounts`: множество счетов со статусом `REVIEW`;
- `rejected_accounts`: множество счетов со статусом `REJECTED`.

Не задавайте имена статусов заранее: структура должна заполняться
по данным.


In [ ]:
stats_by_status = {}
reason_counts = {}
review_accounts = set()
rejected_accounts = set()
# YOUR CODE HERE

for dec in decisions:

  status = dec["status"]
  if status not in stats_by_status:
    stats_by_status[status] = {"count": 0, "amount": 0.0}
  stats_by_status[status]["count"] += 1
  stats_by_status[status]["amount"] += dec["amount"]

  reason = dec["reason"]
  if reason not in reason_counts:
    reason_counts[reason] = 0
  reason_counts[reason] +=1

  if status == "REVIEW":
    review_accounts.add(dec["account"])

  if status == "REJECTED":
    rejected_accounts.add(dec["account"])

print(stats_by_status)
print(reason_counts)
print(review_accounts)
print(rejected_accounts)



{'APPROVED': {'count': 6, 'amount': 19900.0}, 'REVIEW': {'count': 1, 'amount': 2500.0}, 'REJECTED': {'count': 3, 'amount': 2950.0}}
{'ok': 6, 'daily_limit': 1, 'unsupported_kind': 1, 'blocked_account': 1, 'blocked_merchant': 1}
{'A200'}
{'A300', 'A100', 'A400'}


## 6. Очередь и цикл `while`

`for` удобен, когда известна коллекция для обхода. `while` подходит,
когда завершение определяется состоянием: очередь пуста, исчерпан
лимит или достигнута контрольная точка.

Для учебного списка используем `pop(0)`. У больших очередей такая
операция дорогая; с `collections.deque` познакомимся в работе со
стандартной библиотекой.


### Задание 6. Ручная проверка

Оператор может вручную одобрить превышения суммарно не более чем на
3000 RUB. Создайте копии `final_balances`, `daily_spent` и
`review_queue`.

Пока очередь не пуста:

- извлеките первый элемент;
- если его сумма превышает остаток ручного лимита, поместите ID
  текущего и всех оставшихся элементов в `deferred_review_ids` и
  завершите цикл через `break`;
- иначе примените списание или перевод, обновите дневной расход и
  лимит, добавьте ID в `manual_approved_ids`.

Исходная `review_queue` измениться не должна.


In [ ]:
settled_balances = final_balances.copy()
settled_spent = daily_spent.copy()
pending_review = [event.copy() for event in review_queue]
manual_budget = 3_000.0
manual_approved_ids = []
deferred_review_ids = []
# YOUR CODE HERE

while pending_review:
  event = pending_review.pop(0)

  if event["amount"] > manual_budget:
    deferred_review_ids.append(event["id"])
    for i in pending_review:
      deferred_review_ids.append(i["id"])
    break

  manual_budget -= event["amount"]
  manual_approved_ids.append(event["id"])

  match event["kind"]:
    case "DEPOSIT":
      settled_balances[event["account"]] += event["amount"]

    case "REFUND":
      settled_balances[event["account"]] += event["amount"]

    case "PURCHASE":
      settled_balances[event["account"]] -= event["amount"]
      settled_spent[event["account"]] += event["amount"]

    case "WITHDRAWAL":
      settled_balances[event["account"]] -= event["amount"]
      settled_spent[event["account"]] += event["amount"]

    case "TRANSFER":
      settled_balances[event["account"]] -= event["amount"]
      settled_balances[event["counterparty"]] += event["amount"]
      settled_spent[event["account"]] += event["amount"]

    case "CASHOUT":
      settled_balances[event["account"]] -= event["amount"]
      # settled_spent[event["account"]] += event["amount"]

print(settled_balances)
print(settled_spent)
print(pending_review)
print(manual_approved_ids)
print(deferred_review_ids)



{'A100': 27500.0, 'A200': 10000.0, 'A300': 2500.0, 'A400': 5000.0}
{'A100': 3200.0, 'A200': 9000.0, 'A300': 4000.0, 'A400': 0.0}
[]
['E004']
[]


## 7. Сортировка и отчёт

`sorted` умеет сравнивать кортежи: сначала по первому элементу,
затем по второму. Это позволяет построить рейтинг без отдельной
функции-ключа. `enumerate(..., start=1)` добавляет место в рейтинге.


### Задание 7. Отчёт смены

Сформируйте `spending_ranking` — список кортежей `(account, spent)`
по убыванию расхода. Затем создайте строки с номером места и
объедините их переводами строк в `shift_report`.

Первая строка должна быть `Рейтинг дневного расхода:`. Суммы
форматируйте с двумя знаками после запятой и разделителем тысяч.


In [ ]:
spending_ranking = []
ranking_lines = []
shift_report = ""
# YOUR CODE HERE

# расход, счёт
temp_ranking = [(spent, account) for account, spent in settled_spent.items()]
temp_ranking.sort(reverse=True)

# счёт, расход
spending_ranking = [(account, spent) for spent, account in temp_ranking]

ranking_lines = []
for place, (account, spent) in enumerate(spending_ranking, start=1):
    line = f"{place}. {account} — {spent:,.2f} RUB"
    ranking_lines.append(line)

shift_report = "Рейтинг дневного расхода:\n" + "\n".join(ranking_lines)

print(spending_ranking)
print(ranking_lines)
print(shift_report)

[('A200', 9000.0), ('A300', 4000.0), ('A100', 3200.0), ('A400', 0.0)]
['1. A200 — 9,000.00 RUB', '2. A300 — 4,000.00 RUB', '3. A100 — 3,200.00 RUB', '4. A400 — 0.00 RUB']
Рейтинг дневного расхода:
1. A200 — 9,000.00 RUB
2. A300 — 4,000.00 RUB
3. A100 — 3,200.00 RUB
4. A400 — 0.00 RUB


## 8. Итоговый мини-проект

Вторая смена — независимый набор данных. Здесь появляется
**velocity-лимит**: число одобренных списаний со счёта не должно
превышать заданный максимум. Его проверяют после дневного лимита,
но до достаточности баланса.

### Задание 8. Ночная смена

1. Убедитесь, что ID уникальны (`capstone_ids_unique`).
2. Создайте копию балансов, нулевые расходы и счётчики списаний.
3. Обработайте события по правилам задания 4, добавив velocity-
   проверку. Заблокированный merchant проверяется первым.
4. Перевод меняет два баланса, но расход и число списаний относятся
   только к отправителю.
5. Постройте `capstone_status_counts`, списки review/rejected ID,
   множество риск-счетов и `capstone_health` (`LOW`, если баланс
   меньше 3000 RUB, иначе `STABLE`).
6. Сформируйте итоговую строку `capstone_report` заданного формата.

В `capstone_decisions` достаточно ключей `id`, `account`, `status`,
`reason`, `amount`. Операции `REVIEW`/`REJECTED` баланс не меняют.


In [ ]:
capstone_initial_balances = {"C100": 9_000.0, "C200": 4_000.0}
capstone_daily_limits = {"C100": 5_000.0, "C200": 2_500.0}
capstone_velocity_limits = {"C100": 2, "C200": 2}
capstone_blocked_merchants = {"CASINO"}
capstone_events = [
    {"id": "C001", "kind": "purchase", "account": "C100",
     "amount": 1200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C002", "kind": "withdrawal", "account": "C200",
     "amount": 500.0, "merchant": None, "counterparty": None},
    {"id": "C003", "kind": "purchase", "account": "C100",
     "amount": 4100.0, "merchant": "TECH", "counterparty": None},
    {"id": "C004", "kind": "refund", "account": "C200",
     "amount": 200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C005", "kind": "purchase", "account": "C200",
     "amount": 300.0, "merchant": "CASINO", "counterparty": None},
    {"id": "C006", "kind": "transfer", "account": "C200",
     "amount": 1800.0, "merchant": None, "counterparty": "C100"},
    {"id": "C007", "kind": "purchase", "account": "C200",
     "amount": 100.0, "merchant": "CAFE", "counterparty": None},
    {"id": "C008", "kind": "deposit", "account": "C100",
     "amount": 1000.0, "merchant": None, "counterparty": None},
]


In [ ]:
capstone_ids_unique = None
capstone_balances = capstone_initial_balances.copy()
capstone_spent = {account: 0.0 for account in capstone_balances}
capstone_debit_counts = {account: 0 for account in capstone_balances}
capstone_decisions = []

capstone_status_counts = {}
capstone_review_ids = []
capstone_rejected_ids = []
capstone_risk_accounts = set()
capstone_health = {}
capstone_report = ""
# YOUR CODE HERE

# Проверка уникальности
capstone_ids_unique = True
dupl = []
for i in capstone_events:
  if i["id"] in dupl:
    capstone_ids_unique = False
    break
  dupl.append(i["id"])
# print(len(dupl))
# print(capstone_ids_unique)

# Инициализация
capstone_balances = capstone_initial_balances.copy()
capstone_spent = {account: 0.0 for account in capstone_balances}
capstone_debit_counts = {account: 0 for account in capstone_balances}
capstone_decisions = []

for event in capstone_events:

  if event["merchant"] is not None and event["merchant"] in capstone_blocked_merchants:
    capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "REJECTED",
        "reason": "blocked_merchant",
        "amount": event["amount"],
    })
    continue

  match event["kind"]:
    case "DEPOSIT" | "REFUND":
      capstone_balances[event["account"]] += event["amount"]
      capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "APPROVED",
        "reason": "ok",
        "amount": event["amount"],
      })
      continue

    case "PURCHASE" | "WITHDRAWAL" | "TRANSFER" | "CASHOUT":
      if capstone_spent[event["account"]] + event["amount"] > capstone_daily_limits[event["account"]]:
        capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "REVIEW",
        "reason": "daily_limit",
        "amount": event["amount"],
        })
        continue

      if capstone_debit_counts[event["account"]] + 1 > capstone_velocity_limits[event["account"]]:
        capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "REVIEW",
        "reason": "velocity_limit",
        "amount": event["amount"],
        })
        continue

      if capstone_balances[event["account"]] < event["amount"]:
        capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "REJECTED",
        "reason": "insufficient_funds",
        "amount": event["amount"],
        })
        continue


      capstone_balances[event["account"]] -= event["amount"]
      capstone_spent[event["account"]] += event["amount"]
      capstone_debit_counts[event["account"]] += 1

      if event["kind"] == "TRANSFER":
        capstone_balances[event["counterparty"]] += event["amount"]

      capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "APPROVED",
        "reason": "ok",
        "amount": event["amount"]
      })

    case _:
      capstone_decisions.append({
        "id": event["id"],
        "account": event["account"],
        "status": "REJECTED",
        "reason": "unsupported_kind",
        "amount": event["amount"]
      })

for ids in capstone_decisions:
  status = ids["status"]
  capstone_status_counts[status] = capstone_status_counts.get(status, 0) + 1

capstone_review_ids = [i["id"] for i in capstone_decisions if i["status"] == "REVIEW"]
capstone_rejected_ids = [i["id"] for i in capstone_decisions if i["status"] == "REJECTED"]

for account, balance in capstone_balances.items():
    if balance < 3000:
        capstone_risk_accounts.add(account)

for account, balance in capstone_balances.items():
    if balance < 3000:
        capstone_health[account] = "LOW"
    else:
        capstone_health[account] = "STABLE"

capstone_report = (
    f"Обработано событий: {len(capstone_events)}\n"
    f"APPROVED: {capstone_status_counts.get('APPROVED', 0)}, "
    f"REVIEW: {capstone_status_counts.get('REVIEW', 0)}, "
    f"REJECTED: {capstone_status_counts.get('REJECTED', 0)}\n"
    f"На проверке: {capstone_review_ids}\n"
    f"Отклонены: {capstone_rejected_ids}\n"
    f"Риск: {capstone_risk_accounts}"
)

## Вывод и защита

Добавьте после этой ячейки собственный вывод на 7–10 предложений:

1. Почему порядок событий влияет на результат?
2. Почему E004 отправлена на REVIEW, хотя денег на счёте достаточно?
3. Почему E009 не изменила два баланса автоматически?
4. Чем ошибка входных данных отличается от `REJECTED`?
5. Почему для `review_queue` понадобилась копия?
6. Как сработал velocity-лимит во второй смене?
7. Назовите хотя бы один инвариант системы.

### Контрольные вопросы

1. Когда нужен список, а когда множество?
2. Почему ключ словаря должен быть хешируемым?
3. Чем `append` отличается от `extend`?
4. Почему опасно удалять элементы списка внутри `for` по нему?
5. Когда `continue` делает код понятнее?
6. Чем `while` отличается от `for` в задаче с очередью?
7. Как `match` и `if` разделяют разные виды решений?
8. Что означает «не изменять исходные данные»?
9. Почему проверка лимита должна происходить до изменения баланса?
10. Когда comprehension хуже обычного цикла?


## Ответы на вопросы:

1. Каждая операция опирается на результаты предыдущих. Баланс и дневной расход меняются по ходу обработки, а лимиты проверяются относительно текущего состояния

2. Потому что в правилах проверка дневного лимита идёт раньше проверки баланса. У счёта A200 после E003 уже израсходовано 6500 RUB, а дневной лимит A200 — 8000 RUB. Добавление E004 на 2500 RUB даёт 9000 RUB > 8000 RUB — лимит превышен. Денег на счету хватает, но это не важно: проверка баланса до этой операции просто не доходит. Событие уходит на ручную проверку, а не отклоняется.

3. Потому что E009 — это TRANSFER со счёта A300 на A200, а счёт A300 заблокирован (он в blocked_accounts)

4. Ошибка входных данных — это структурный брак, из-за которого запись вообще нельзя считать операцией: дубликат ID, некорректная сумма, неизвестный счёт или получатель. REJECTED — это уже бизнес-решение по корректной записи: данные валидны, но правила шлюза запрещают операцию

5. Потому что исходный список review_queue из задания 4 — это результат работы процессора, и его нельзя портить. В задании 6 мы извлекаем элементы через pop(0) и меняем балансы; если бы мы делали это с оригиналом, то потеряли бы исходную очередь и не смогли бы её проверить/пересчитать.

6. Velocity-лимит ограничивает число одобренных списаний со счёта. В ночной смене он проверяется после дневного лимита, но до проверки баланса. Счётчик capstone_debit_counts увеличивается только при APPROVED-списаниях (purchase/withdrawal/transfer/cashout), причём у перевода — только у отправителя. Как только число списаний превышает лимит, следующее списание уходит в REVIEW с причиной velocity_limit, даже если денег достаточно. Так шлюз ограничивает частоту, а не только сумму.

7. Сумма балансов не меняется от переводов: при TRANSFER баланс отправителя уменьшается ровно на столько, на сколько увеличивается баланс получателя. Общая сумма денег в системе сохраняется.

_Напишите здесь собственный вывод._
